# Taller: demanda de pasajeros en estaciones de TransMilenio

**Pregunta:** ¿cuántas personas entran a una estación troncal en cada franja de 15 minutos, según la hora, el tipo de día y la estación?

**Estaciones de estudio:** Alcalá, Modelia y Terreros. **Periodo:** junio a agosto de 2026.

Estructura del notebook (una sección por fase del taller):

| Bloque | Contenido | Responsable |
|---|---|---|
| 0 | Configuración | |
| 1 | Fase 1: problema y base de datos | |
| 2 | Fase 2: exploración y preparación | |
| 3 | Fase 3: métodos y matriz de aplicabilidad | |
| 4 | Fase 4: comparación de finalistas | |
| 5 | Fase 5: insumos para los dos informes | |

## Bloque 0. Configuración
Todo lo que se puede cambiar está en esta celda. El resto del notebook no tiene valores escritos a mano.

In [ ]:
from pathlib import Path
from datetime import datetime, timedelta
import warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, RegressorMixin, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer

SEED = 42
np.random.seed(SEED)
warnings.filterwarnings("ignore", category=UserWarning)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW, PROC, FIG = ROOT / "data" / "raw", ROOT / "data" / "processed", ROOT / "reports" / "figuras"
for p in (RAW, PROC, FIG):
    p.mkdir(parents=True, exist_ok=True)

# Estaciones de estudio: código -> nombre corto. Con None se usan todas las estaciones troncales.
ESTACIONES = {"02200": "Alcalá", "06001": "Modelia", "07504": "Terreros"}

# Festivos del periodo (confirmar con un calendario oficial)
FESTIVOS = pd.to_datetime(["2026-06-08", "2026-06-15", "2026-06-29",
                           "2026-07-20", "2026-08-07", "2026-08-17"])

DIAS_TEST = 21        # los últimos días del periodo se reservan como conjunto de prueba
N_FOLDS = 5           # validación cruzada por grupos de fecha
RECARGAR = False      # True = volver a leer los Excel aunque exista el archivo procesado

## Bloque 1. Fase 1: problema y base de datos

**Para redactar (entregable de una página):**

- **Origen:** TransMilenio S.A., "Validaciones mensuales del SITP por franja horaria", Datos Abiertos Bogotá. Enlace y fecha de descarga: _completar_.
- **Tamaño:** _completar con la salida de la celda 1.2_.
- **Variable objetivo:** validaciones por estación, fecha y franja de 15 minutos.
- **Variables explicativas:** _completar con la tabla de la celda 2.4_.
- **Por qué este problema:** _completar_.

### 1.1 Función de carga
Cada Excel trae una fila por acceso y franja, y una columna por día. La función lo pasa a una fila por estación, fecha y franja.

In [ ]:
def cargar_mes(ruta):
    """Lee un Excel mensual de validaciones troncales y devuelve (tabla larga, registros crudos)."""
    bruto = pd.read_excel(ruta, sheet_name=0, header=None)
    fila = bruto.index[bruto.eq("Fase").any(axis=1)][0]            # fila del encabezado
    enc = bruto.loc[fila].tolist()
    c0 = enc.index("Fase")
    primera = next(h for h in enc if isinstance(h, (datetime, pd.Timestamp)))
    c_dia = enc.index(primera)
    c_tot = next((i for i, h in enumerate(enc) if isinstance(h, str) and "Total" in h), len(enc))
    fechas = [pd.Timestamp(primera) + timedelta(days=i) for i in range(c_tot - c_dia)]

    df = bruto.iloc[fila + 1:, c0:c_tot].copy()
    df.columns = ["fase", "linea", "estacion", "acceso", "intervalo"] + fechas
    df = df[df.fase.notna() & (df.fase != "Total general")]
    df = df[df.linea.str.contains("Zona", na=False)]              # estaciones troncales (no rutas duales)
    df["codigo"] = df.estacion.str.extract(r"\((\d+)\)")[0]
    crudos = len(df) * len(fechas)                                 # registros acceso x franja x día
    if ESTACIONES is not None:
        df = df[df.codigo.isin(ESTACIONES)]

    largo = df.melt(id_vars=["codigo", "linea", "acceso", "intervalo"], value_vars=fechas,
                    var_name="fecha", value_name="validaciones")
    largo["validaciones"] = pd.to_numeric(largo.validaciones, errors="coerce")
    largo["franja_min"] = largo.intervalo.apply(lambda t: t.hour * 60 + t.minute)
    out = (largo.groupby(["codigo", "linea", "fecha", "franja_min"], as_index=False)
                .agg(validaciones=("validaciones", "sum"), accesos=("acceso", "nunique")))
    return out, crudos

### 1.2 Cargar todos los meses de `data/raw/`

In [ ]:
archivo_proc = PROC / "validaciones_modelo.csv"
if archivo_proc.exists() and not RECARGAR:
    datos = pd.read_csv(archivo_proc, dtype={"codigo": str}, parse_dates=["fecha"])
    print("Leído de", archivo_proc.name)
else:
    partes, resumen = [], []
    for ruta in sorted(RAW.glob("*.xlsx")):
        tabla, crudos = cargar_mes(ruta)
        partes.append(tabla)
        resumen.append({"archivo": ruta.name[:60], "registros_crudos": crudos,
                        "registros_modelo": len(tabla),
                        "desde": tabla.fecha.min().date(), "hasta": tabla.fecha.max().date()})
    assert partes, f"No hay archivos .xlsx en {RAW}"
    datos = pd.concat(partes, ignore_index=True).sort_values(["codigo", "fecha", "franja_min"])
    datos.to_csv(archivo_proc, index=False)
    resumen = pd.DataFrame(resumen)
    resumen.to_csv(PROC / "resumen_carga.csv", index=False)
    display(resumen)

nombres = ESTACIONES or {c: c for c in datos.codigo.unique()}
datos["estacion"] = datos.codigo.map(nombres)
print("Registros para modelar:", len(datos), "| estaciones:", datos.codigo.nunique(),
      "| días:", datos.fecha.nunique(), "| de", datos.fecha.min().date(), "a", datos.fecha.max().date())
datos.head()

### 1.3 Oferta programada (opcional)
Si `data/raw/gtfs.zip` existe, agrega los buses y rutas programados por estación, tipo de día y franja. Si no existe, la celda no hace nada.

In [ ]:
ruta_gtfs = RAW / "gtfs.zip"
oferta = None
if ruta_gtfs.exists():
    z = zipfile.ZipFile(ruta_gtfs)
    stops = pd.read_csv(z.open("stops.txt"), dtype=str)
    trips = pd.read_csv(z.open("trips.txt"), dtype=str, usecols=["route_id", "service_id", "trip_id"])
    cal = pd.read_csv(z.open("calendar.txt"), dtype=str)
    st = pd.read_csv(z.open("stop_times.txt"),
                     usecols=["trip_id", "departure_time", "stop_id", "stop_sequence"],
                     dtype={"trip_id": str, "departure_time": str, "stop_id": str, "stop_sequence": int})
    stops["est"] = stops.parent_station.fillna(stops.stop_id)
    codigos = {str(int(c)): c for c in datos.codigo.unique()}          # GTFS usa el código sin cero inicial
    st["codigo"] = st.stop_id.map(dict(zip(stops.stop_id, stops.est))).map(codigos)
    st["ultima"] = st.groupby("trip_id").stop_sequence.transform("max")
    st = st[st.codigo.notna() & (st.stop_sequence < st.ultima)]        # sin la parada final del viaje
    st = st.drop_duplicates(["trip_id", "codigo"]).merge(trips, on="trip_id")

    dias = cal.melt(id_vars="service_id", value_vars=["monday", "saturday", "sunday"],
                    var_name="d", value_name="activo").query("activo == '1'")
    dias["tipo_dia"] = dias.d.map({"monday": "habil", "saturday": "sabado", "sunday": "domingo_festivo"})
    st = st.merge(dias[["service_id", "tipo_dia"]], on="service_id")
    hms = st.departure_time.str.split(":", expand=True).astype(int)
    st["franja_min"] = (hms[0] * 60 + hms[1]) // 15 * 15
    oferta = (st.groupby(["codigo", "tipo_dia", "franja_min"])
                .agg(buses=("trip_id", "nunique"), rutas=("route_id", "nunique")).reset_index())
    print("Oferta calculada:", oferta.shape, "| estaciones:", oferta.codigo.nunique())
else:
    print("Sin gtfs.zip: se modela solo con las validaciones.")

## Bloque 2. Fase 2: exploración y preparación

### 2.1 Calidad de los datos: nulos, duplicados, ceros

In [ ]:
print("Nulos por columna:"); print(datos.isna().sum())
print("\nDuplicados (estación, fecha, franja):", datos.duplicated(["codigo", "fecha", "franja_min"]).sum())
print("Registros con cero validaciones: {:.1%}".format((datos.validaciones == 0).mean()))
display(datos.groupby("estacion").validaciones.describe().round(1))

### 2.2 Distribución de la variable objetivo

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
datos.validaciones.plot.hist(bins=60, ax=ax[0], title="Validaciones por franja")
np.log1p(datos.validaciones).plot.hist(bins=60, ax=ax[1], title="log(1 + validaciones)")
plt.tight_layout(); plt.savefig(FIG / "distribucion_objetivo.png", dpi=120); plt.show()
print("Asimetría original: {:.2f} | en logaritmo: {:.2f}".format(
      datos.validaciones.skew(), np.log1p(datos.validaciones).skew()))

### 2.3 Variables derivadas

In [ ]:
datos["hora"] = datos.franja_min / 60
datos["dia_semana"] = datos.fecha.dt.dayofweek
datos["festivo"] = datos.fecha.isin(FESTIVOS).astype(int)
datos["tipo_dia"] = np.select(
    [(datos.dia_semana == 6) | (datos.festivo == 1), datos.dia_semana == 5],
    ["domingo_festivo", "sabado"], default="habil")
datos["franja"] = datos.franja_min.apply(lambda m: f"{m // 60:02d}:{m % 60:02d}")
datos["grupo"] = datos.estacion + "|" + datos.tipo_dia               # para splines y kernels
datos["celda"] = datos.grupo + "|" + datos.franja_min.astype(str)    # para Ridge y Lasso
datos["y"] = np.log1p(datos.validaciones)

NUMERICAS = ["accesos"]
if oferta is not None:
    datos = datos.merge(oferta, on=["codigo", "tipo_dia", "franja_min"], how="left")
    datos[["buses", "rutas"]] = datos[["buses", "rutas"]].fillna(0)
    NUMERICAS += ["buses", "rutas"]
datos[["estacion", "fecha", "hora", "tipo_dia", "validaciones", "y"] + NUMERICAS].head()

### 2.4 Relación con la hora y con el tipo de día
Esta gráfica es la evidencia principal para decidir si splines y kernels aportan.

In [ ]:
perfil = datos.groupby(["estacion", "tipo_dia", "hora"]).validaciones.mean().reset_index()
est = sorted(datos.estacion.unique())[:6]
fig, ax = plt.subplots(1, len(est), figsize=(4.5 * len(est), 3.5), sharey=False, squeeze=False)
for a, e in zip(ax[0], est):
    for t, g in perfil[perfil.estacion == e].groupby("tipo_dia"):
        a.plot(g.hora, g.validaciones, label=t)
    a.set_title(e); a.set_xlabel("Hora"); a.set_ylabel("Validaciones promedio"); a.legend()
plt.tight_layout(); plt.savefig(FIG / "perfil_horario.png", dpi=120); plt.show()

### 2.5 Correlaciones entre variables numéricas
Sirve para argumentar si Ridge y Lasso tienen multicolinealidad que tratar.

In [ ]:
display(datos[["y", "hora", "dia_semana", "festivo"] + NUMERICAS].corr().round(2))

### 2.6 Decisiones de limpieza (completar y justificar)

| Decisión | Qué se hizo | Por qué |
|---|---|---|
| Filas vacías y "Total general" | Se eliminan en la carga | No son observaciones |
| Rutas duales (línea sin "Zona") | Se excluyen | Son buses en vía, no estaciones |
| Ceros | _completar_ | _completar_ |
| Outliers | _completar_ | _completar_ |
| Transformación del objetivo | log(1 + y) | _completar con la asimetría de 2.2_ |

### 2.7 Partición entrenamiento / prueba
Se parte **por fecha**: los últimos días son la prueba. Partir al azar mezclaría el mismo día en los dos conjuntos e inflaría las métricas.

In [ ]:
corte = datos.fecha.max() - pd.Timedelta(days=DIAS_TEST)
train, test = datos[datos.fecha <= corte].copy(), datos[datos.fecha > corte].copy()
X_train, y_train, grupos_train = train, train.y, train.fecha
X_test, y_test = test, test.y
print("Entrenamiento:", len(train), "registros,", train.fecha.nunique(), "días, hasta", corte.date())
print("Prueba:       ", len(test), "registros,", test.fecha.nunique(), "días")

## Bloque 3. Fase 3: métodos y matriz de aplicabilidad
Todos los hiperparámetros se eligen con validación cruzada **solo sobre entrenamiento**, con los pliegues formados por fecha. Las métricas de este bloque están en escala logarítmica; en el bloque 4 se pasan a pasajeros.

### 3.0 Herramientas comunes

In [ ]:
cv = GroupKFold(n_splits=min(N_FOLDS, train.fecha.nunique()))
SCORING = {"rmse": "neg_root_mean_squared_error", "mae": "neg_mean_absolute_error", "r2": "r2"}
resultados, modelos, hiper = [], {}, []

def ajustar(nombre, modelo, grid):
    """Busca hiperparámetros por validación cruzada y guarda el resultado."""
    gs = GridSearchCV(modelo, grid, cv=cv, scoring=SCORING, refit="rmse", n_jobs=-1)
    gs.fit(X_train, y_train, groups=grupos_train)
    r, i = gs.cv_results_, gs.best_index_
    resultados.append({"modelo": nombre,
                       "RMSE_cv": -r["mean_test_rmse"][i], "RMSE_sd": r["std_test_rmse"][i],
                       "MAE_cv": -r["mean_test_mae"][i], "R2_cv": r["mean_test_r2"][i]})
    for p, valores in grid.items():
        if len(valores) > 1:
            elegido = gs.best_params_[p]
            borde = elegido in (valores[0], valores[-1])
            hiper.append({"modelo": nombre, "hiperparametro": p.split("__")[-1],
                          "rango_probado": f"{valores[0]:.4g} a {valores[-1]:.4g} ({len(valores)} valores)",
                          "elegido": elegido, "en_el_borde": borde})
            if borde:
                print(f"AVISO {nombre}: {p}={elegido} quedó en el borde del rango; ampliar la búsqueda.")
    modelos[nombre] = gs
    print(f"{nombre}: RMSE cv = {-r['mean_test_rmse'][i]:.4f} ± {r['std_test_rmse'][i]:.4f}")
    return gs

def columnas(cat, num, extra=None):
    pasos = [("cat", OneHotEncoder(handle_unknown="ignore"), cat),
             ("num", StandardScaler(), num)]
    return ColumnTransformer(pasos + (extra or []))

class SplinePorGrupo(BaseEstimator, TransformerMixin):
    """Base de splines de la hora, una curva distinta por grupo (estación y tipo de día)."""
    def __init__(self, n_knots=8, degree=3):
        self.n_knots, self.degree = n_knots, degree
    def fit(self, X, y=None):
        self.sp_ = SplineTransformer(n_knots=self.n_knots, degree=self.degree,
                                     extrapolation="constant").fit(X[["hora"]].to_numpy())
        self.grupos_ = np.array(sorted(X["grupo"].unique()))
        return self
    def transform(self, X):
        B = self.sp_.transform(X[["hora"]].to_numpy())
        G = (X["grupo"].to_numpy()[:, None] == self.grupos_[None, :]).astype(float)
        return (B[:, :, None] * G[:, None, :]).reshape(len(X), -1)

class NadarayaWatson(BaseEstimator, RegressorMixin):
    """Suavización por kernel gaussiano sobre la hora, por grupo. h = ancho de banda en horas."""
    def __init__(self, h=0.25):
        self.h = h
    def fit(self, X, y):
        t = pd.DataFrame({"grupo": X["grupo"].to_numpy(), "hora": X["hora"].to_numpy(), "y": np.asarray(y)})
        self.media_ = t.y.mean()
        self.tabla_ = {g: d.groupby("hora").y.agg(["sum", "count"]) for g, d in t.groupby("grupo")}
        return self
    def predict(self, X):
        out = np.full(len(X), self.media_)
        hora, grupo = X["hora"].to_numpy(), X["grupo"].to_numpy()
        for g in np.unique(grupo):
            if g not in self.tabla_:
                continue
            m, tab = grupo == g, self.tabla_[g]
            W = np.exp(-0.5 * ((hora[m][:, None] - tab.index.to_numpy()[None, :]) / self.h) ** 2)
            den = W @ tab["count"].to_numpy()
            out[m] = np.where(den > 0, (W @ tab["sum"].to_numpy()) / np.maximum(den, 1e-300), self.media_)
        return out

### 3.1 Línea base: predecir la media
Referencia mínima. Un modelo que no la supere no aporta.

In [ ]:
ajustar("Base (media)", DummyRegressor(strategy="mean"), {});

### 3.2 Regresión lineal (OLS)
Modelo de referencia: la hora entra como una recta.

In [ ]:
ols = Pipeline([("prep", columnas(["estacion", "tipo_dia"], ["hora"] + NUMERICAS)),
                ("reg", LinearRegression())])
ajustar("OLS", ols, {});

### 3.3 Ridge
Diseño amplio: una variable por cada combinación de estación, tipo de día y franja, más las numéricas. Se incluye el mismo diseño con OLS para que la comparación sea justa.

In [ ]:
prep_amplio = columnas(["celda", "dia_semana"], NUMERICAS)
ajustar("OLS diseño amplio", Pipeline([("prep", prep_amplio), ("reg", LinearRegression())]), {})
ajustar("Ridge", Pipeline([("prep", prep_amplio), ("reg", Ridge())]),
        {"reg__alpha": list(np.logspace(-3, 3, 13))});

### 3.4 Lasso
Mismo diseño amplio. Además de regularizar, deja coeficientes en cero.

In [ ]:
gs_lasso = ajustar("Lasso", Pipeline([("prep", prep_amplio), ("reg", Lasso(max_iter=20000))]),
                   {"reg__alpha": list(np.logspace(-5, -1, 9))})
coef = gs_lasso.best_estimator_.named_steps["reg"].coef_
print(f"Coeficientes en cero: {(coef == 0).sum()} de {len(coef)}")

### 3.5 Splines
Una curva suave de la hora por cada estación y tipo de día. El hiperparámetro es el número de nudos.

In [ ]:
prep_sp = ColumnTransformer([("sp", SplinePorGrupo(), ["hora", "grupo"]),
                             ("cat", OneHotEncoder(handle_unknown="ignore"), ["dia_semana"]),
                             ("num", StandardScaler(), NUMERICAS)])
ajustar("Splines", Pipeline([("prep", prep_sp), ("reg", LinearRegression())]),
        {"prep__sp__n_knots": [4, 6, 8, 12, 16, 24, 32]});

### 3.6 Kernels
Suavización de Nadaraya-Watson sobre la hora, por estación y tipo de día. El hiperparámetro es el ancho de banda (en horas). Con una sola variable no hace falta estandarizar; si se agregan más, sí.

In [ ]:
ajustar("Kernel (Nadaraya-Watson)", NadarayaWatson(),
        {"h": [0.05, 0.1, 0.15, 0.25, 0.4, 0.6, 1.0, 1.5]});

### 3.7 Resultados de validación cruzada e hiperparámetros

In [ ]:
tabla_cv = pd.DataFrame(resultados).set_index("modelo").round(4).sort_values("RMSE_cv")
tabla_hiper = pd.DataFrame(hiper)
tabla_cv.to_csv(PROC / "resultados_cv.csv"); tabla_hiper.to_csv(PROC / "hiperparametros.csv", index=False)
display(tabla_cv); display(tabla_hiper)

### 3.8 Matriz de aplicabilidad (completar con criterio propio)

| Método | ¿Aporta? | Justificación estadística | Resultado (RMSE cv) |
|---|---|---|---|
| Regresión lineal (OLS) | | | |
| Ridge | | | |
| Lasso | | | |
| Splines | | | |
| Kernels | | | |

Preguntas guía: ¿la relación con la hora es una recta (2.4)? ¿Ridge mejora al OLS del mismo diseño por más que la desviación entre pliegues? ¿Cuántos coeficientes anula Lasso? ¿Splines y kernels se diferencian entre sí?

## Bloque 4. Fase 4: comparación de finalistas
Los tres mejores por validación cruzada se evalúan una sola vez sobre el conjunto de prueba, en **pasajeros**. La incertidumbre se estima con bootstrap remuestreando días completos.

In [ ]:
finalistas = [m for m in tabla_cv.index if not m.startswith("Base")][:3]
print("Finalistas:", finalistas)

real = test.validaciones.to_numpy()
pred = {m: np.clip(np.expm1(modelos[m].predict(X_test)), 0, None) for m in finalistas + ["Base (media)"]}

def metricas(y, p):
    return {"MAE": mean_absolute_error(y, p), "RMSE": mean_squared_error(y, p) ** 0.5, "R2": r2_score(y, p)}

tabla_test = pd.DataFrame({m: metricas(real, p) for m, p in pred.items()}).T.round(2)
display(tabla_test)

In [ ]:
rng = np.random.default_rng(SEED)
dias_test = test.fecha.unique()
idx_dia = {d: np.flatnonzero((test.fecha == d).to_numpy()) for d in dias_test}
B, boot = 500, {m: [] for m in finalistas}
for _ in range(B):
    idx = np.concatenate([idx_dia[d] for d in rng.choice(dias_test, len(dias_test))])
    for m in finalistas:
        boot[m].append(mean_absolute_error(real[idx], pred[m][idx]))
boot = pd.DataFrame(boot)

ic = boot.quantile([0.025, 0.5, 0.975]).T.round(2)
ic.columns = ["MAE_2.5%", "MAE_mediana", "MAE_97.5%"]
display(ic)
mejor = ic["MAE_mediana"].idxmin()
for m in finalistas:
    if m != mejor:
        d = (boot[m] - boot[mejor]).quantile([0.025, 0.975]).round(2).tolist()
        print(f"{m} menos {mejor}: diferencia de MAE entre {d[0]} y {d[1]}",
              "(empate)" if d[0] <= 0 <= d[1] else "(diferencia real)")
tabla_test.to_csv(PROC / "resultados_test.csv"); ic.to_csv(PROC / "bootstrap_mae.csv")

### 4.1 Real contra predicho en un día de prueba

In [ ]:
dia = sorted(dias_test)[len(dias_test) // 2]
est = sorted(test.estacion.unique())[:6]
fig, ax = plt.subplots(1, len(est), figsize=(4.5 * len(est), 3.5), squeeze=False)
for a, e in zip(ax[0], est):
    m = ((test.fecha == dia) & (test.estacion == e)).to_numpy()
    a.plot(test.hora[m], real[m], "k.", label="Real")
    a.plot(test.hora[m], pred[mejor][m], label=mejor)
    a.set_title(f"{e}, {pd.Timestamp(dia).date()}"); a.set_xlabel("Hora"); a.legend()
plt.tight_layout(); plt.savefig(FIG / "real_vs_predicho.png", dpi=120); plt.show()

### 4.2 Decisión final (completar)

- **Ganador y por qué:** _métrica, variabilidad, interpretabilidad, costo de usarlo_.
- **¿Hubo empate dentro del margen de ruido?** _si sí, qué criterio lo desempató_.
- **Limitaciones:** _tres estaciones, un trimestre, festivos, oferta programada frente a la real_.

## Bloque 5. Fase 5: insumos para los informes

**Informe técnico** (`reports/informe_tecnico.md`): tablas en `data/processed/` (`resumen_carga`, `resultados_cv`, `hiperparametros`, `resultados_test`, `bootstrap_mae`) y figuras en `reports/figuras/`.

**Informe gerencial** (`reports/informe_gerencial.md`): la celda siguiente traduce el error a pasajeros por estación y, si hay oferta, lo compara con los buses programados.

In [ ]:
test = test.assign(prediccion=pred[mejor], error=np.abs(real - pred[mejor]))
resumen_g = test.groupby("estacion").agg(
    entradas_dia=("validaciones", lambda s: s.sum() / test.fecha.nunique()),
    error_promedio_por_franja=("error", "mean")).round(1)
display(resumen_g)

pico = (test[test.tipo_dia == "habil"].groupby(["estacion", "franja"])
        .agg(esperado=("prediccion", "mean"), **({"buses": ("buses", "mean")} if "buses" in test else {}))
        .reset_index().sort_values("esperado", ascending=False).groupby("estacion").head(3))
if "buses" in pico:
    pico["pasajeros_por_bus"] = (pico.esperado / pico.buses.where(pico.buses > 0)).round(0)
display(pico.round(1))
resumen_g.to_csv(PROC / "resumen_gerencial.csv")